# TOR-Like Relay Node

Each node: listens → decrypts → forwards → encrypts response back

In [ ]:
import socket
import threading
import ssl
from time import sleep
from cryptography.fernet import Fernet


# Hardcoded session keys
key0 = b'2kVu8vEYFzxGtBW2RUvHQTGi8kQBqCx5uOYi6YqW3HM='
key1 = b'u1fHL7dGkj8NvwP2sE9mOYRTu5jLZCx4yNZX7WqV3IK='
key2 = b'm9yKJ6hFij7OxP1rF8nLZQSv6kMCDw3zOAXU8YpW2JN='


def encrypt_message(message, session_key):
    f = Fernet(session_key)
    if isinstance(message, str):
        message = message.encode()
    return f.encrypt(message)


def decrypt_message(encrypted_message, session_key):
    f = Fernet(session_key)
    return f.decrypt(encrypted_message)


class Node:
    PORT_START = 5000
    
    def __init__(self, id, next_node=None, session_key=None):
        self.id = id
        self.port = self.PORT_START + id
        self.next_node = next_node
        self.session_key = session_key
        self.running = True

    def handle_client(self, conn, addr):
        try:
            conn.settimeout(3.0)
            
            data = b""
            while True:
                try:
                    chunk = conn.recv(4096)
                    if not chunk:
                        break
                    data += chunk
                except socket.timeout:
                    break
            
            if not data:
                return
            
            decrypted = decrypt_message(data, self.session_key)
            
            if self.next_node:
                response = self.send_to_next(decrypted)
                encrypted = encrypt_message(response, self.session_key)
                conn.sendall(encrypted)
            else:
                host = self.extract_host(decrypted)
                context = ssl.create_default_context()
                
                with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                    with context.wrap_socket(s, server_hostname=host) as ssl_s:
                        ssl_s.settimeout(10.0)
                        ssl_s.connect((host, 443))
                        ssl_s.sendall(decrypted)
                        
                        response = b""
                        while True:
                            chunk = ssl_s.recv(4096)
                            if not chunk:
                                break
                            response += chunk
                
                encrypted = encrypt_message(response, self.session_key)
                conn.sendall(encrypted)
                
        except Exception as e:
            print(f"Node {self.id} error: {e}")
        finally:
            conn.close()

    def extract_host(self, request_bytes):
        request_str = request_bytes.decode("utf-8")
        lines = request_str.split("\r\n")
        for line in lines:
            if line.startswith("Host: "):
                return line.split(" ")[1]
        return None

    def send_to_next(self, data):
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(10.0)
        sock.connect(("127.0.0.1", self.next_node.port))
        sock.sendall(data)
        
        response = b""
        while True:
            try:
                chunk = sock.recv(4096)
                if not chunk:
                    break
                response += chunk
            except socket.timeout:
                break
        sock.close()
        return response

    def start(self):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("127.0.0.1", self.port))
        s.listen(5)
        print(f"Node {self.id} listening on port {self.port}")
        
        while self.running:
            try:
                conn, addr = s.accept()
                t = threading.Thread(target=self.handle_client, args=(conn, addr), daemon=True)
                t.start()
            except:
                break
        s.close()


print("Creating nodes...")
node0 = Node(id=0, session_key=key0)
node1 = Node(id=1, next_node=node0, session_key=key1)
node2 = Node(id=2, next_node=node1, session_key=key2)

print("Starting node threads...")
threading.Thread(target=node0.start, daemon=True).start()
threading.Thread(target=node1.start, daemon=True).start()
threading.Thread(target=node2.start, daemon=True).start()

print("Waiting for nodes to initialize...")
sleep(2)

print("\n=== Sending HTTP request through circuit ===")
http_request = b"GET / HTTP/1.1\r\nHost: example.com\r\nConnection: close\r\n\r\n"

l0 = encrypt_message(http_request, key0)
l1 = encrypt_message(l0, key1)
onion = encrypt_message(l1, key2)

print(f"Original: {len(http_request)} bytes")
print(f"Encrypted: {len(onion)} bytes")

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(15.0)
    sock.connect(("127.0.0.1", 5002))
    sock.sendall(onion)
    
    response = b""
    while True:
        try:
            chunk = sock.recv(8192)
            if not chunk:
                break
            response += chunk
        except socket.timeout:
            break
    
    print(f"Received: {len(response)} bytes")
    
    if response:
        r1 = decrypt_message(response, key2)
        r2 = decrypt_message(r1, key1)
        final = decrypt_message(r2, key0)
        
        print(f"Decrypted: {len(final)} bytes")
        print("\n=== Response from example.com ===")
        print(final[:500].decode("utf-8", errors="ignore"))
    else:
        print("ERROR: No response received!")
